# 01 — Common Data Quality Audit

**ผู้รับผิดชอบ:** ทอฝัน  
**วันที่ตรวจ:** 18 สิงหาคม 2026 (ปรับปรุงโค้ดให้อ่านง่ายสไตล์ Data Science ปี 4)  
**ขอบเขต:** ตรวจสอบคุณภาพข้อมูลที่ผ่านการทำความสะอาดแล้วทั้ง 5 ตารางใน `data/processed/` เปรียบเทียบกับ `data/raw/`

Notebook นี้ตรวจสอบความถูกต้องของ Schema, ค่าว่าง (Missing Values), ข้อมูลซ้ำ (Duplicates), Primary Key, Foreign Key, ความสอดคล้องข้ามตาราง (Cross-table Consistency), ช่วงของค่า (Domain Validity) และตรรกะทางเวลา

In [2]:
# 1. Setup และ Import ไลบรารีที่จำเป็น
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# กำหนด Path หลักของโปรเจกต์
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from kaverentai.data.load_data import load_all_data, TABLES
from kaverentai.data.validate_data import validate_tables

# โหลดข้อมูลทั้งแบบ Cleaned (Processed) และ Raw เพื่อนำมาเปรียบเทียบ
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

tables = load_all_data(cleaned=True, data_dir=PROCESSED_DIR)
raw_tables = load_all_data(cleaned=False, data_dir=RAW_DIR)

# ตั้งค่าการแสดงผลตารางของ Pandas ให้อ่านง่าย
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

print(f"Project Root: {PROJECT_ROOT}")
print(f"โหลดข้อมูลสำเร็จ 5 ตาราง: {list(tables.keys())}")

Project Root: d:\final project_
โหลดข้อมูลสำเร็จ 5 ตาราง: ['projects', 'units', 'listings', 'leases', 'weekly_market']


## 1. ภาพรวมและ Schema ขั้นพื้นฐาน

ตรวจสอบจำนวนแถว จำนวนคอลัมน์ แถวที่ซ้ำซ้อน และขนาดหน่วยความจำของแต่ละตาราง พร้อมตรวจ Schema เบื้องต้น

In [3]:
# ตรวจสอบ Schema ตามข้อกำหนดใน validate_data.py
schema_errors = validate_tables(tables)

# สร้างตารางสรุปภาพรวม (Table Inventory)
inventory_data = []
for name, df in tables.items():
    inventory_data.append({
        'Table': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Duplicate_Rows': df.duplicated().sum(),
        'Memory_MB': df.memory_usage(deep=True).sum() / (1024 ** 2)
    })

df_inventory = pd.DataFrame(inventory_data).set_index('Table')
display(df_inventory.style.format({'Rows': '{:,}', 'Memory_MB': '{:.2f}'}))

if not schema_errors:
    print("ผลตรวจ Schema: PASS (ทุกตารางมีคอลัมน์ครบถ้วนและไม่เป็นตารางว่าง)")
else:
    print("ผลตรวจ Schema: FAIL")
    display(schema_errors)

,Rows,Columns,Duplicate_Rows,Memory_MB
Table,,,,
projects,19,7,0,0.00
units,"15,367",8,0,3.57
listings,"31,544",24,0,21.65
leases,"50,965",10,0,13.00
weekly_market,"2,288",9,0,0.37


ผลตรวจ Schema: PASS (ทุกตารางมีคอลัมน์ครบถ้วนและไม่เป็นตารางว่าง)


## 2. พจนานุกรมข้อมูล (Data Dictionary)

สรุปรายละเอียดของแต่ละคอลัมน์: ชนิดข้อมูล (Data Type), จำนวนค่าที่ไม่ว่าง, และจำนวนค่าที่ไม่ซ้ำ (Unique Values)

In [4]:
# คำอธิบายความหมายของแต่ละคอลัมน์
COLUMN_DESCRIPTIONS = {
    'project_id': 'รหัสโครงการ (PK: projects)',
    'project_name': 'ชื่อโครงการคอนโดมิเนียม',
    'university': 'มหาวิทยาลัย/ทำเลหลักของโครงการ',
    'distance_to_campus_m': 'ระยะห่างจากมหาวิทยาลัย (เมตร)',
    'total_units': 'จำนวนห้องชุดทั้งหมดในโครงการ',
    'year_completed': 'ปีที่โครงการสร้างเสร็จ',
    'facility_count': 'จำนวนสิ่งอำนวยความสะดวกส่วนกลาง',
    'unit_id': 'รหัสห้องชุด (PK: units)',
    'floor': 'ชั้นของห้องพัก (1-8)',
    'size_sqm': 'ขนาดพื้นที่ห้องชุด (ตร.ม.)',
    'room_type': 'ประเภทห้อง (studio, 1BR, 1BR Plus)',
    'view': 'วิวห้องพัก (standard, city, pool)',
    'furnished': 'มีเฟอร์นิเจอร์ครบ (True/False)',
    'agent_id': 'รหัสนายหน้าที่ดูแล',
    'listing_id': 'รหัสประกาศปล่อยเช่า (PK: listings)',
    'week_listed': 'สัปดาห์ที่ลงประกาศ',
    'date_listed': 'วันที่ลงประกาศ (YYYY-MM-DD)',
    'season_listed': 'ฤดูกาล ณ วันที่ลงประกาศ',
    'asking_rent': 'ราคาตั้งเช่าล่าสุด (บาท/เดือน)',
    'first_asking_rent': 'ราคาตั้งเช่าครั้งแรก (บาท/เดือน)',
    'weeks_on_market': 'จำนวนสัปดาห์ที่ประกาศอยู่ในตลาด',
    'n_viewings': 'จำนวนครั้งที่มีผู้เข้าชมห้อง',
    'leased': 'ปล่อยเช่าสำเร็จแล้วหรือไม่ (True/False)',
    'week_leased': 'สัปดาห์ที่ปล่อยเช่าได้ (ว่างเมื่อยังปล่อยไม่ได้)',
    'tenant_segment': 'กลุ่มผู้เช่า (ว่างเมื่อยังปล่อยไม่ได้)',
    'lease_months': 'ระยะเวลาสัญญาเช่าเป็นเดือน (ว่างเมื่อยังปล่อยไม่ได้)',
    'lease_id': 'รหัสสัญญาเช่า (PK: leases)',
    'week_start': 'สัปดาห์ที่เริ่มสัญญา',
    'date_start': 'วันที่เริ่มสัญญาเช่า',
    'week_end': 'สัปดาห์ที่สิ้นสุดสัญญา',
    'rent': 'ค่าเช่าจริงตามสัญญา (บาท/เดือน)',
    'is_renewal': 'เป็นการต่อสัญญาเดิมหรือไม่ (True/False)',
    'week': 'ลำดับสัปดาห์ (PK: weekly_market ร่วมกับ project_id)',
    'date': 'วันที่เริ่มต้นสัปดาห์',
    'season': 'ฤดูกาลของสัปดาห์นั้น',
    'n_searchers': 'จำนวนผู้ค้นหาห้องในสัปดาห์นั้น',
    'n_listings_open': 'จำนวนประกาศที่เปิดว่างอยู่',
    'n_units_leased': 'จำนวนห้องที่มีผู้เช่าอยู่ ณ ตอนนั้น',
    'occupancy': 'อัตราการเข้าพัก (0 ถึง 1)',
    'median_asking_rent': 'ราคากลางค่าเช่าของตลาดสัปดาห์นั้น'
}

# รวมข้อมูลโครงสร้างของแต่ละตาราง
dict_list = []
for table_name, df in tables.items():
    for col in df.columns:
        dict_list.append({
            'Table': table_name,
            'Column': col,
            'DataType': str(df[col].dtype),
            'Non-Null Count': df[col].notna().sum(),
            'Null Count': df[col].isna().sum(),
            'Unique Values': df[col].nunique(),
            'Description': COLUMN_DESCRIPTIONS.get(col, '-')
        })

df_dict = pd.DataFrame(dict_list)
display(df_dict.style.format({'Non-Null Count': '{:,}', 'Null Count': '{:,}', 'Unique Values': '{:,}'}))

,Table,Column,DataType,Non-Null Count,Null Count,Unique Values,Description
0,projects,project_id,int64,19,0,19,รหัสโครงการ (PK: projects)
1,projects,project_name,object,19,0,19,ชื่อโครงการคอนโดมิเนียม
2,projects,university,object,19,0,11,มหาวิทยาลัย/ทำเลหลักของโครงการ
3,projects,distance_to_campus_m,int64,19,0,12,ระยะห่างจากมหาวิทยาลัย (เมตร)
4,projects,total_units,int64,19,0,18,จำนวนห้องชุดทั้งหมดในโครงการ
5,projects,year_completed,int64,19,0,9,ปีที่โครงการสร้างเสร็จ
6,projects,facility_count,int64,19,0,9,จำนวนสิ่งอำนวยความสะดวกส่วนกลาง
7,units,unit_id,object,"15,367",0,"15,367",รหัสห้องชุด (PK: units)
8,units,project_id,int64,"15,367",0,19,รหัสโครงการ (PK: projects)
9,units,floor,int64,"15,367",0,8,ชั้นของห้องพัก (1-8)


## 3. การตรวจสอบค่าว่าง (Missing Values) และ Business Logic

ในเชิงสถิติ ค่าว่างไม่ได้แปลว่าข้อมูลผิดพลาดเสมอไป โดยมีเงื่อนไขทางธุรกิจ (Business Rules) ดังนี้:
1. คอลัมน์ `week_leased`, `tenant_segment`, `lease_months` จะต้องว่าง **เฉพาะห้องที่ยังปล่อยไม่ได้ (`leased == False`) เท่านั้น**
2. คอลัมน์ `median_asking_rent` จะต้องว่าง **เฉพาะสัปดาห์ที่ไม่มีประกาศเปิดให้เช่า (`n_listings_open == 0`) เท่านั้น**

In [5]:
# 1. สรุปภาพรวมค่าว่างในแต่ละตาราง
missing_summary = []
for name, df in tables.items():
    null_counts = df.isna().sum()
    for col, count in null_counts[null_counts > 0].items():
        missing_summary.append({
            'Table': name,
            'Column': col,
            'Missing Count': count,
            'Missing Percentage': f"{(count / len(df)) * 100:.2f}%"
        })

print("=== สรุปคอลัมน์ที่มีค่าว่าง ===")
display(pd.DataFrame(missing_summary))

# 2. ตรวจสอบตรรกะทางธุรกิจในตาราง listings
listings = tables['listings']
outcome_cols = ['week_leased', 'tenant_segment', 'lease_months']

# ก) ห้องที่ปล่อยได้แล้ว (leased == True) ต้องไม่มีค่าว่างใน outcome_cols
leased_missing = listings[listings['leased']][outcome_cols].isna().sum().sum()

# ข) ห้องที่ยังปล่อยไม่ได้ (leased == False) ต้องว่างทั้งหมดใน outcome_cols
unleased_not_null = listings[~listings['leased']][outcome_cols].notna().sum().sum()

# 3. ตรวจสอบตรรกะทางธุรกิจในตาราง weekly_market
weekly = tables['weekly_market']
# median_asking_rent ที่ว่าง ต้องเกิดขึ้นเฉพาะสัปดาห์ที่ n_listings_open == 0
invalid_median_null = weekly[weekly['median_asking_rent'].isna()]['n_listings_open'].gt(0).sum()

# สรุปผลการตรวจตรรกะค่าว่าง
missing_rules = pd.DataFrame([
    {'Rule': 'ห้องที่เช่าสำเร็จแล้ว ต้องมีข้อมูลผลลัพธ์ครบ (ไม่มีค่าว่าง)', 'Failures': leased_missing},
    {'Rule': 'ห้องที่ยังปล่อยไม่ได้ ต้องไม่มีข้อมูลผลลัพธ์ (ต้องเป็นค่าว่าง)', 'Failures': unleased_not_null},
    {'Rule': 'median_asking_rent ว่างได้เฉพาะตอนไม่มีประกาศเปิด (n_listings_open = 0)', 'Failures': invalid_median_null}
])
missing_rules['Status'] = np.where(missing_rules['Failures'] == 0, 'PASS', 'FAIL')
display(missing_rules)

=== สรุปคอลัมน์ที่มีค่าว่าง ===


,Table,Column,Missing Count,Missing Percentage
0,listings,week_leased,566,1.79%
1,listings,tenant_segment,566,1.79%
2,listings,lease_months,566,1.79%
3,weekly_market,median_asking_rent,3,0.13%


,Rule,Failures,Status
0,ห้องที่เช่าสำเร็จแล้ว ต้องมีข้อมูลผลลัพธ์ครบ (ไม่มีค่าว่าง),0,PASS
1,ห้องที่ยังปล่อยไม่ได้ ต้องไม่มีข้อมูลผลลัพธ์ (ต้องเป็นค่าว่าง),0,PASS
2,median_asking_rent ว่างได้เฉพาะตอนไม่มีประกาศเปิด (n_listings_open = 0),0,PASS


## 4. ตรวจสอบ Primary Key, Foreign Key และความสอดคล้องข้ามตาราง (Cross-table Consistency)

ตรวจสอบความสมบูรณ์เชิงสัมพันธ์ของฐานข้อมูล (Relational Integrity):
- Primary Key ของแต่ละตารางต้องไม่ซ้ำกัน (Unique)
- รหัสอ้างอิง (Foreign Key) ต้องชี้ไปยังตารางต้นทางที่มีอยู่จริง
- ข้อมูลของห้องชุด (เช่น ขนาดห้อง, ชั้น, วิว) ในตาราง `listings` ต้องตรงกับตาราง `units`

In [6]:
projects = tables['projects']
units = tables['units']
leases = tables['leases']

integrity_checks = []

# --- 1. ตรวจสอบ Primary Key ---
integrity_checks.append({'Category': 'Primary Key', 'Check': 'projects.project_id ไม่ซ้ำ', 'Failures': projects['project_id'].duplicated().sum()})
integrity_checks.append({'Category': 'Primary Key', 'Check': 'units.unit_id ไม่ซ้ำ', 'Failures': units['unit_id'].duplicated().sum()})
integrity_checks.append({'Category': 'Primary Key', 'Check': 'listings.listing_id ไม่ซ้ำ', 'Failures': listings['listing_id'].duplicated().sum()})
integrity_checks.append({'Category': 'Primary Key', 'Check': 'leases.lease_id ไม่ซ้ำ', 'Failures': leases['lease_id'].duplicated().sum()})
integrity_checks.append({'Category': 'Primary Key', 'Check': 'weekly_market.(project_id + week) ไม่ซ้ำ', 'Failures': weekly.duplicated(['project_id', 'week']).sum()})

# --- 2. ตรวจสอบ Foreign Key ---
integrity_checks.append({'Category': 'Foreign Key', 'Check': 'units.project_id อยู่ใน projects', 'Failures': (~units['project_id'].isin(projects['project_id'])).sum()})
integrity_checks.append({'Category': 'Foreign Key', 'Check': 'listings.project_id อยู่ใน projects', 'Failures': (~listings['project_id'].isin(projects['project_id'])).sum()})
integrity_checks.append({'Category': 'Foreign Key', 'Check': 'listings.unit_id อยู่ใน units', 'Failures': (~listings['unit_id'].isin(units['unit_id'])).sum()})
integrity_checks.append({'Category': 'Foreign Key', 'Check': 'leases.project_id อยู่ใน projects', 'Failures': (~leases['project_id'].isin(projects['project_id'])).sum()})
integrity_checks.append({'Category': 'Foreign Key', 'Check': 'leases.unit_id อยู่ใน units', 'Failures': (~leases['unit_id'].isin(units['unit_id'])).sum()})
integrity_checks.append({'Category': 'Foreign Key', 'Check': 'weekly_market.project_id อยู่ใน projects', 'Failures': (~weekly['project_id'].isin(projects['project_id'])).sum()})

# --- 3. ตรวจสอบความสอดคล้องข้ามตาราง (Cross-table Consistency) ---
merged_check = listings[['unit_id', 'floor', 'size_sqm', 'room_type']].merge(
    units[['unit_id', 'floor', 'size_sqm', 'room_type']],
    on='unit_id',
    suffixes=('_listing', '_unit')
)
mismatched_specs = (
    (merged_check['floor_listing'] != merged_check['floor_unit']) |
    (merged_check['size_sqm_listing'] != merged_check['size_sqm_unit']) |
    (merged_check['room_type_listing'] != merged_check['room_type_unit'])
).sum()
integrity_checks.append({'Category': 'Cross-table', 'Check': 'คุณสมบัติห้องใน listings ตรงกับ units 100%', 'Failures': mismatched_specs})

df_integrity = pd.DataFrame(integrity_checks)
df_integrity['Status'] = np.where(df_integrity['Failures'] == 0, 'PASS', 'FAIL')
display(df_integrity)

,Category,Check,Failures,Status
0,Primary Key,projects.project_id ไม่ซ้ำ,0,PASS
1,Primary Key,units.unit_id ไม่ซ้ำ,0,PASS
2,Primary Key,listings.listing_id ไม่ซ้ำ,0,PASS
3,Primary Key,leases.lease_id ไม่ซ้ำ,0,PASS
4,Primary Key,weekly_market.(project_id + week) ไม่ซ้ำ,0,PASS
5,Foreign Key,units.project_id อยู่ใน projects,0,PASS
6,Foreign Key,listings.project_id อยู่ใน projects,0,PASS
7,Foreign Key,listings.unit_id อยู่ใน units,0,PASS
8,Foreign Key,leases.project_id อยู่ใน projects,0,PASS
9,Foreign Key,leases.unit_id อยู่ใน units,0,PASS


## 5. ตรวจสอบช่วงของค่า (Domain Validity) และตรรกะทางธุรกิจ

ตรวจสอบค่าตัวแปรตามข้อกำหนดทางธุรกิจและความสมเหตุสมผลทางสถิติ:
- คอนโดเป็น Low-rise ดังนั้น ชั้นต้องอยู่ในช่วง 1–8
- ขนาดห้องอยู่ในช่วงปกติ (20–50 ตร.ม.)
- ราคาค่าเช่าและจำนวนสัปดาห์ต้องไม่ติดลบ
- สัปดาห์ที่ปล่อยเช่าได้ (`week_leased`) ต้องเท่ากับ `week_listed + weeks_on_market` พอดี

In [7]:
domain_checks = []

# 1. ตาราง units
domain_checks.append({'Target': 'units', 'Check': 'ชั้น (floor) อยู่ระหว่าง 1–8', 'Failures': (~units['floor'].between(1, 8)).sum()})
domain_checks.append({'Target': 'units', 'Check': 'ขนาดห้อง (size_sqm) อยู่ระหว่าง 20–50 ตร.ม.', 'Failures': (~units['size_sqm'].between(20, 50)).sum()})
domain_checks.append({'Target': 'units', 'Check': 'room_type อยู่ในหมวด (studio, 1BR, 1BR Plus)', 'Failures': (~units['room_type'].isin(['studio', '1BR', '1BR Plus'])).sum()})
domain_checks.append({'Target': 'units', 'Check': 'view อยู่ในหมวด (standard, city, pool)', 'Failures': (~units['view'].isin(['standard', 'city', 'pool'])).sum()})

# 2. ตาราง listings
domain_checks.append({'Target': 'listings', 'Check': 'ราคาตั้งเช่า (first_asking_rent) > 0', 'Failures': (listings['first_asking_rent'] <= 0).sum()})
domain_checks.append({'Target': 'listings', 'Check': 'weeks_on_market >= 0 (ไม่ติดลบ)', 'Failures': (listings['weeks_on_market'] < 0).sum()})
domain_checks.append({'Target': 'listings', 'Check': 'n_viewings >= 0 (ไม่ติดลบ)', 'Failures': (listings['n_viewings'] < 0).sum()})
domain_checks.append({'Target': 'listings', 'Check': 'season_listed ถูกต้อง (normal, peak, shoulder, off)', 'Failures': (~listings['season_listed'].isin(['normal', 'peak', 'shoulder', 'off'])).sum()})

# 3. ตรรกะสัปดาห์ที่ปล่อยเช่าได้: week_leased == week_listed + weeks_on_market
leased_subset = listings[listings['leased']]
calc_week_leased = leased_subset['week_listed'] + leased_subset['weeks_on_market']
week_mismatch = (leased_subset['week_leased'] != calc_week_leased).sum()
domain_checks.append({'Target': 'listings', 'Check': 'week_leased = week_listed + weeks_on_market', 'Failures': week_mismatch})

# 4. ตรวจสอบการปรับราคาขึ้น (asking_rent > first_asking_rent)
rent_increased = (listings['asking_rent'] > listings['first_asking_rent']).sum()
domain_checks.append({
    'Target': 'listings',
    'Check': 'asking_rent <= first_asking_rent (สังเกตการณ์: พบรอบ -B ที่ราคาเพิ่ม)',
    'Failures': rent_increased
})

# 5. ตาราง weekly_market
domain_checks.append({'Target': 'weekly_market', 'Check': 'occupancy อยู่ระหว่าง 0 ถึง 1', 'Failures': (~weekly['occupancy'].between(0, 1)).sum()})
domain_checks.append({'Target': 'weekly_market', 'Check': 'n_searchers และ n_listings_open ไม่ติดลบ', 'Failures': ((weekly['n_searchers'] < 0) | (weekly['n_listings_open'] < 0)).sum()})

df_domain = pd.DataFrame(domain_checks)
# ข้อสังเกตเรื่อง rent_increased ให้เป็น WARN ส่วนข้ออื่นเป็น PASS/FAIL
df_domain['Status'] = np.where(df_domain['Failures'] == 0, 'PASS', 'WARN')
display(df_domain)

,Target,Check,Failures,Status
0,units,ชั้น (floor) อยู่ระหว่าง 1–8,0,PASS
1,units,ขนาดห้อง (size_sqm) อยู่ระหว่าง 20–50 ตร.ม.,0,PASS
2,units,"room_type อยู่ในหมวด (studio, 1BR, 1BR Plus)",0,PASS
3,units,"view อยู่ในหมวด (standard, city, pool)",0,PASS
4,listings,ราคาตั้งเช่า (first_asking_rent) > 0,0,PASS
5,listings,weeks_on_market >= 0 (ไม่ติดลบ),0,PASS
6,listings,n_viewings >= 0 (ไม่ติดลบ),0,PASS
7,listings,"season_listed ถูกต้อง (normal, peak, shoulder, off)",0,PASS
8,listings,week_leased = week_listed + weeks_on_market,0,PASS
9,listings,asking_rent <= first_asking_rent (สังเกตการณ์: พบรอบ -B ที่ราคาเพิ่ม),285,WARN


## 6. ตรวจสอบความถูกต้องของวันที่และลำดับสัปดาห์ (Date & Week Consistency)

ในโปรเจกต์นี้กำหนดสัปดาห์ที่ 0 เริ่มต้นที่ **2023-01-03**  
ดังนั้น วันที่ที่บันทึกไว้ในทุกตารางจะต้องเท่ากับ `2023-01-03 + (7 × week)` เสมอ

In [8]:
BASE_DATE = pd.Timestamp('2023-01-03')

date_targets = [
    ('listings', listings, 'date_listed', 'week_listed'),
    ('leases', leases, 'date_start', 'week_start'),
    ('weekly_market', weekly, 'date', 'week')
]

date_summary = []
for name, df, date_col, week_col in date_targets:
    parsed_date = pd.to_datetime(df[date_col], errors='coerce')
    expected_date = BASE_DATE + pd.to_timedelta(df[week_col] * 7, unit='D')
    
    invalid_dates = parsed_date.isna().sum()
    mismatches = (parsed_date != expected_date).sum()
    
    date_summary.append({
        'Table': name,
        'Date Column': date_col,
        'Min Date': parsed_date.min().strftime('%Y-%m-%d'),
        'Max Date': parsed_date.max().strftime('%Y-%m-%d'),
        'Invalid Dates': invalid_dates,
        'Date-Week Mismatches': mismatches
    })

df_date = pd.DataFrame(date_summary)
df_date['Status'] = np.where((df_date['Invalid Dates'] == 0) & (df_date['Date-Week Mismatches'] == 0), 'PASS', 'FAIL')
display(df_date)

,Table,Date Column,Min Date,Max Date,Invalid Dates,Date-Week Mismatches,Status
0,listings,date_listed,2023-01-03,2026-06-23,0,0,PASS
1,leases,date_start,2023-01-03,2026-06-23,0,0,PASS
2,weekly_market,date,2023-01-03,2026-06-23,0,0,PASS


## 7. เปรียบเทียบข้อมูลก่อนและหลังทำความสะอาด (Raw vs Processed)

ตรวจสอบว่าขั้นตอน Data Cleaning ไม่ทำแถวสูญหาย (`row_count_preserved`) และได้แก้ไขจุดบกพร่องที่พบในตารางดิบเรียบร้อยแล้ว เช่น ตัดช่องว่างหัวท้ายข้อความ แปลงตัวพิมพ์เล็ก และจัดรูปแบบวันที่

In [9]:
# 1. เปรียบเทียบจำนวนแถวและจำนวนเซลล์ว่างทั้งหมด
compare_list = []
for name in TABLES:
    df_raw = raw_tables[name]
    df_clean = tables[name]
    compare_list.append({
        'Table': name,
        'Raw Rows': len(df_raw),
        'Processed Rows': len(df_clean),
        'Row Count Match': len(df_raw) == len(df_clean),
        'Raw Null Cells': df_raw.isna().sum().sum(),
        'Processed Null Cells': df_clean.isna().sum().sum()
    })

df_compare = pd.DataFrame(compare_list)
display(df_compare.style.format({'Raw Rows': '{:,}', 'Processed Rows': '{:,}', 'Raw Null Cells': '{:,}', 'Processed Null Cells': '{:,}'}))

# 2. เปรียบเทียบการแก้ไขปัญหาสำคัญเฉพาะในตาราง listings
raw_l = raw_tables['listings']
clean_l = tables['listings']

cleanup_issues = pd.DataFrame([
    {
        'Issue Description': 'size_sqm อยู่นอกช่วง 20–50 ตร.ม.',
        'Raw (จำนวนที่พบ)': (~raw_l['size_sqm'].between(20, 50)).sum(),
        'Processed (หลัง Clean)': (~clean_l['size_sqm'].between(20, 50)).sum()
    },
    {
        'Issue Description': 'room_type มีช่องว่างเกินหัว/ท้าย (Trailing whitespace)',
        'Raw (จำนวนที่พบ)': (raw_l['room_type'] != raw_l['room_type'].astype(str).str.strip()).sum(),
        'Processed (หลัง Clean)': (clean_l['room_type'] != clean_l['room_type'].astype(str).str.strip()).sum()
    },
    {
        'Issue Description': 'view ไม่เป็นตัวพิมพ์เล็ก (Inconsistent casing)',
        'Raw (จำนวนที่พบ)': (raw_l['view'] != raw_l['view'].astype(str).str.lower()).sum(),
        'Processed (หลัง Clean)': (clean_l['view'] != clean_l['view'].astype(str).str.lower()).sum()
    },
    {
        'Issue Description': 'date_listed ไม่ใช่รูปแบบมาตรฐาน YYYY-MM-DD',
        'Raw (จำนวนที่พบ)': (~raw_l['date_listed'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$')).sum(),
        'Processed (หลัง Clean)': (~clean_l['date_listed'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$')).sum()
    }
])

display(cleanup_issues.style.format({'Raw (จำนวนที่พบ)': '{:,}', 'Processed (หลัง Clean)': '{:,}'}))

,Table,Raw Rows,Processed Rows,Row Count Match,Raw Null Cells,Processed Null Cells
0,projects,19,19,True,0,0
1,units,"15,367","15,367",True,0,0
2,listings,"31,544","31,544",True,"6,835","1,698"
3,leases,"50,965","50,965",True,0,0
4,weekly_market,"2,288","2,288",True,3,3


,Issue Description,Raw (จำนวนที่พบ),Processed (หลัง Clean)
0,size_sqm อยู่นอกช่วง 20–50 ตร.ม.,63,0
1,room_type มีช่องว่างเกินหัว/ท้าย (Trailing whitespace),"1,598",0
2,view ไม่เป็นตัวพิมพ์เล็ก (Inconsistent casing),"1,591",0
3,date_listed ไม่ใช่รูปแบบมาตรฐาน YYYY-MM-DD,"31,544",0


## 8. สรุปผลการตรวจสอบคุณภาพข้อมูล (Final Quality Audit Summary)

สรุปผลรวมทุกเกณฑ์การตรวจสอบคุณภาพข้อมูลทั้ง 5 ตาราง เพื่อส่งต่อไปยังขั้นตอน EDA และการสร้างแบบจำลอง

In [10]:
# นับจำนวนผลการทดสอบทั้งหมด
total_fails = (df_inventory['Duplicate_Rows'] > 0).sum() + \
              (missing_rules['Status'] == 'FAIL').sum() + \
              (df_integrity['Status'] == 'FAIL').sum() + \
              (df_domain['Status'] == 'FAIL').sum() + \
              (df_date['Status'] == 'FAIL').sum()

total_warns = (df_domain['Status'] == 'WARN').sum()
unleased_count = (~listings['leased']).sum()

overall_status = 'PASS' if total_fails == 0 else 'FAIL'
display(Markdown(f"""
### บทสรุปผลการตรวจสอบ: **{overall_status} (ผ่านการตรวจสอบสมบูรณ์)**

- **ข้อผิดพลาดร้ายแรง (Critical Errors):** {total_fails} รายการ
- **ข้อสังเกตเพิ่มเติม (Warnings / Observations):** {total_warns} รายการ (พบการปรับราคาขึ้นในรอบ -B ซึ่งควรตระหนักไว้ตอนทำ Feature)
- **ความสัมพันธ์ของฐานข้อมูล:** Primary Key ไม่ซ้ำ และ Foreign Key เชื่อมโยงกันถูกต้องสมบูรณ์ 100%
- **การจัดการค่าว่าง:** ค่าว่างจำนวน **{unleased_count:,} แถว** ใน `listings` เป็นห้องที่ยังปล่อยเช่าไม่ได้ (**Right-Censored Data**) ถูกต้องตาม Business Logic ห้ามลบทิ้ง
- **ความพร้อมในการใช้งาน:** ข้อมูลใน `data/processed/` มีความสะอาด ถูกต้อง และพร้อมสำหรับการนำไปทำ **EDA** และพัฒนาโมเดล **Rental Price** และ **Market & Occupancy Forecasting** ต่อไป โดยต้องตรวจนิยาม occupancy และความครบถ้วนของช่วงเวลาสำหรับการพยากรณ์เพิ่มเติม
"""))


### บทสรุปผลการตรวจสอบ: **PASS (ผ่านการตรวจสอบสมบูรณ์)**

- **ข้อผิดพลาดร้ายแรง (Critical Errors):** 0 รายการ
- **ข้อสังเกตเพิ่มเติม (Warnings / Observations):** 1 รายการ (พบการปรับราคาขึ้นในรอบ -B ซึ่งควรตระหนักไว้ตอนทำ Feature)
- **ความสัมพันธ์ของฐานข้อมูล:** Primary Key ไม่ซ้ำ และ Foreign Key เชื่อมโยงกันถูกต้องสมบูรณ์ 100%
- **การจัดการค่าว่าง:** ค่าว่างจำนวน **566 แถว** ใน `listings` เป็นห้องที่ยังปล่อยเช่าไม่ได้ (**Right-Censored Data**) ถูกต้องตาม Business Logic ห้ามลบทิ้ง
- **ความพร้อมในการใช้งาน:** ข้อมูลใน `data/processed/` มีความสะอาด ถูกต้อง และพร้อมสำหรับการนำไปทำ **EDA** และพัฒนาโมเดล **Rental Price** และ **Market & Occupancy Forecasting** ต่อไป โดยต้องตรวจนิยาม occupancy และความครบถ้วนของช่วงเวลาสำหรับการพยากรณ์เพิ่มเติม
